# This method attemps to gain a higher accuracy by training a ResNet model on Amazon-Berkley Objects dataset since the ABO dataset contains object images and their respective weight

# Download Dataset

In [2]:
import os

# 1. Download and extract listings metadata (approx 83 MB)
if not os.path.exists('listings'):
    print("Downloading listings metadata...")
    !wget https://amazon-berkeley-objects.s3.amazonaws.com/archives/abo-listings.tar
    !tar -xf abo-listings.tar
else:
    print("Listings metadata already exists, skipping download...")

# 2. Download and extract downscaled images (approx 3 GB)
if not os.path.exists('images'):
    print("Downloading downscaled images...")
    !wget https://amazon-berkeley-objects.s3.amazonaws.com/archives/abo-images-small.tar
    !tar -xf abo-images-small.tar
else:
    print("Images already exist, skipping download...")

Listings metadata already exists, skipping download...
Images already exist, skipping download...


In [3]:
import torch
import torchvision.models as models
from torch import nn
import torch.nn.functional as F
from typing import Dict, Union, List

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
import pandas as pd

# Load the listings metadata
df = pd.read_json('listings/metadata/listings_0.json.gz', lines=True, compression='gzip')

# Load the images metadata
images_df = pd.read_csv('images/metadata/images.csv.gz', compression='gzip')

# Join metadata with images using image_id
# For main images:
df_with_main_image = df.merge(
    images_df,
    left_on='main_image_id',
    right_on='image_id',
    how='left'
)

# View the result
print(df_with_main_image[['item_id', 'item_name', 'main_image_id', 'image_id']].head())

# Check shape and data types
# print(df.shape)
# print(df.info())
# Print first 10 rows
# print(df.head(10))
# print(df_with_main_image[['item_id', 'item_name', 'main_image_id', 'image_id']])


      item_id                                          item_name  \
0  B06X9STHNG  [{'language_tag': 'nl_NL', 'value': 'Amazon-me...   
1  B07P8ML82R  [{'language_tag': 'es_MX', 'value': '22" Botto...   
2  B07H9GMYXS  [{'language_tag': 'en_AE', 'value': 'AmazonBas...   
3  B07CTPR73M  [{'language_tag': 'en_GB', 'value': 'Stone & B...   
4  B01MTEI8M6  [{'language_tag': 'en_AU', 'value': 'The Fix A...   

  main_image_id     image_id  
0   81iZlv3bjpL  81iZlv3bjpL  
1   619y9YG9cnL  619y9YG9cnL  
2   81NP7qh2L6L  81NP7qh2L6L  
3   61Rp4qOih9L  61Rp4qOih9L  
4   714CmIfKIYL  714CmIfKIYL  
